In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-generative" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
from abstractgraph.utils import plot_embedding_2d
def _compute_Z(model, graphs, targets):
    try:
        import umap.umap_ as umap
        from sklearn.neighbors import NeighborhoodComponentsAnalysis
        import numpy as np

        embs = model.transform(graphs)
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r".*n_jobs value .* overridden .* by setting random_state.*",
                category=UserWarning,
                module="umap.umap_",
            )
            Z_umap = umap.UMAP(n_components=10, random_state=42, init="random").fit_transform(embs)
        y = np.asarray(targets)
        # Fit NCA only on classes 0 and 1 (exclude class 2), then transform all
        fit_mask = (y == 0) | (y == 1)
        nca = NeighborhoodComponentsAnalysis(n_components=2, random_state=42)
        nca.fit(Z_umap[fit_mask], y[fit_mask])
        Z = nca.transform(Z_umap)
        return Z
    except Exception:
        return None
    
def plot_2d_scatter(model, graphs, targets, alpha=.65):
    size = 6
    Z_all = _compute_Z(model, graphs, targets)
    fig, axes = plt.subplots(1, 3, figsize=(3.1*size, size), constrained_layout=True)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix="All data", mode="scatter", alpha=alpha, ax=axes[0], show=False, Z=Z_all)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix="All data", mode="class_union", k=5, alpha=alpha, z=1, ax=axes[1], show=False, show_instances=False, Z=Z_all)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix="All data", mode="class_union", k=11, alpha=alpha, z=1, ax=axes[2], show=False, show_instances=False, Z=Z_all)
    plt.show()

def estimated_2d_scatter(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets):
    from NSPPK.nsppk import NSPPK 
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

    for _target_id in np.unique(test_targets):
        target_generated_graphs = [g for g, t in zip(generated_graphs, generated_targets) if t == _target_id]
        print(f'#original graphs: {len(test_graphs)}    #generated graphs for target id {_target_id}: {len(target_generated_graphs)}')
        plot_2d_scatter(vectorizer,test_graphs + target_generated_graphs,list(test_targets) + [max(test_targets) + 1] * len(target_generated_graphs))

In [ ]:
def estimated_predictive_performance(train_graphs, train_targets, test_graphs, test_targets):
    from NSPPK.nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)
    train_X = vectorizer.transform(train_graphs)
    test_X = vectorizer.transform(test_graphs)
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    clf.fit(train_X, train_targets)
    from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
    y_prob = clf.predict_proba(test_X)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(test_targets, y_pred)
    print(f'Test accuracy: {acc:.3f} | ROC-AUC: {roc_auc_score(test_targets, y_prob):.3f} | AvgPrec: {average_precision_score(test_targets, y_prob):.3f}')

In [ ]:
def estimated_generative_quality(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets):
    from NSPPK.nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

    from sklearn.ensemble import ExtraTreesClassifier
    classifier = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42)

    from abstractgraph_generative.generative_performance import compute_expected_gain_weighted_equivalent_data_size
    results = compute_expected_gain_weighted_equivalent_data_size(
        generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets,
        vectorizer=vectorizer, classifier=classifier,
        fracional_size=(1,2,3,4,5,7,10,15),
        n_repeats=30,
    )
    print(f'Expected gain weighted equivalent data size:{results["expected_gain_weighted_equivalent_data_size"]:.3f}')

    from abstractgraph_generative.generative_performance import plot_expected_gain_weighted_equivalent_data_size
    _ = plot_expected_gain_weighted_equivalent_data_size(results)

---

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[1]
#assay_id = '488975' #active 2,634
size = int(2000/.33) #to account for 0.33 for test and 0.5 for reference
print(f"size: {size}")
use_equalized = True


limit_active = int(size // 2) if use_equalized else int(size)
limit_inactive = int(size // 2) if use_equalized else int(size)
graphs, targets = loader.load(
    assay_id,
    limit_active=limit_active,
    limit_inactive=limit_inactive,
)
targets = np.array(targets)

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:
from sklearn.model_selection import train_test_split
test_size = 0.33
all_train_graphs, test_graphs, all_train_targets, test_targets = train_test_split(
    graphs,
    targets,
    test_size=test_size,
    stratify=targets if len(np.unique(targets)) > 1 else None,
    random_state=0,
)
reference_split_size = 0.5
train_graphs, reference_graphs, train_targets, reference_targets = train_test_split(
    all_train_graphs,
    all_train_targets,
    test_size=reference_split_size,
    stratify=all_train_targets if len(np.unique(all_train_targets)) > 1 else None,
    random_state=0,
)
print(f"AID {assay_id} graphs: {len(graphs)} (train={len(train_graphs)}, reference={len(reference_graphs)}, test={len(test_graphs)})")
estimated_predictive_performance(train_graphs, train_targets, test_graphs, test_targets)

In [ ]:

from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator, FeasibilityEstimatorNumberOfNodesInRange
from abstractgraph_generative.interpolate import InterpolationEstimator
from abstractgraph_generative.interpolation import InterpolationGenerator
from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import RandomForestClassifier
from abstractgraph.vectorize import AbstractGraphTransformer

nbits=14

decomposition_function = add(
    neighborhood(radius=(1,2)),
    graphlet(radius=2, number_of_nodes=5),
    cycle(),
    tree(),
)


df = neighborhood(radius=1)
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

df = cycle()
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

df = compose(neighborhood(radius=3), unlabel())
fe3 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

min_size, max_size = np.quantile([len(g) for g in train_graphs], [0.25, 0.75])
fe4 = FeasibilityEstimatorNumberOfNodesInRange(min_size=min_size, max_size=max_size)

feasibility_estimators = [fe1, fe2, fe3, fe4]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

graph_transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=decomposition_function,
    return_dense=True,
)

interpolation_estimator = InterpolationEstimator(
    graph_transformer=graph_transformer,
    n_iterations=2,
    n_samples=10,
    k=5,
    feasibility_estimator=feasibility_estimator,
    degree_penalty="auto",
    degree_penalty_mode="multiplicative",
)

generator = InterpolationGenerator(interpolation_estimator).fit(train_graphs, train_targets)

In [ ]:
%%time
from abstractgraph_graphicalizer.chem import draw_molecules

generated_graphs, generated_targets = generator.generate(
    n_rounds=1,
    n_pairs=30,
    n_iterations=3,
    best_of=2,
    prob_for_acceptance_interval=(0.51, 0.95),
    verbose=True,
    draw_func=draw_molecules,
)

In [ ]:
estimated_2d_scatter(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets)

In [ ]:
estimated_generative_quality(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets)

---